# T4 (bin-pick with 60 deg wrist reorientation) data collection

Healthy and anomaly recording via RTDE at 125 Hz. Payload 2 kg.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
# 30 cycles. Same trajectory, same speeds, same 60 deg reorientation as NB1d.
# Gripper: 2.0 kg only (remove the added weight first). Pendant: 2.0 kg.
# This file becomes the matched-HOME healthy reference for the A2 anomaly comparison.

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
q0   = rtde_r.getActualQ()
print(f"TCP[0]: x={tcp0[0]:+.3f} y={tcp0[1]:+.3f} z={tcp0[2]:+.3f}  rx={tcp0[3]:+.3f} ry={tcp0[4]:+.3f} rz={tcp0[5]:+.3f}")
print(f"q[0]:   J1={q0[0]:+.3f} J2={q0[1]:+.3f} J3={q0[2]:+.3f} J4={q0[3]:+.3f} J5={q0[4]:+.3f} J6={q0[5]:+.3f}")
print()
print("Verify: gripper has ONLY 2.0 kg baseline (added 2.0 kg removed).")
print("Verify: pendant declares 2.0 kg.")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_TALL   = (180/25.4, 140/25.4)
FIG_DOUBLE = (180/25.4, 70/25.4)

ROOT        = Path(r"D:\Research\RCM")
LAB_DATA    = ROOT / "Lab_Data" / "T4_BinReorient" / "healthy"
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
HOME_REF    = ROOT / "Lab_Data" / "T4_BinReorient" / "T4_HOME_reference.json"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LAB_DATA, LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB1d_T4_Healthy_session2"
TASK     = "T4"
SESSION  = "session2"
HZ       = 125
N_SCRIPT = 35
N_ACCEPT = 30

PAYLOAD_PENDANT_KG  = 2.0
PAYLOAD_PHYSICAL_KG = 2.0

V     = 0.06
A     = 0.25
SLEEP = 0.3
ROT   = 1.0472   # +60 deg about TCP Z

SCRIPT_PORT = 30002

def send_urscript(script: str):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect((ROBOT_IP, SCRIPT_PORT))
    s.send((script + "\n").encode())
    s.close()

def build_t4(N):
    return f"""
def task_t4():
  local v  = {V}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above  = pose_trans(home, p[ 0.10, 0.00,  0.05, 0,0, 0])
  local pick        = pose_trans(home, p[ 0.10, 0.00, -0.05, 0,0, 0])
  local place_above = pose_trans(home, p[-0.10, 0.10,  0.05, 0,0, {ROT}])
  local place       = pose_trans(home, p[-0.10, 0.10, -0.05, 0,0, {ROT}])

  while (i < N):
    movel(pick_above,  a=a, v=v)
    movel(pick,        a=a, v=v*0.7)
    sleep(dt)
    movel(pick_above,  a=a, v=v*0.7)

    movel(place_above, a=a, v=v)
    movel(place,       a=a, v=v*0.7)
    sleep(dt)
    movel(place_above, a=a, v=v*0.7)

    movel(home, a=a, v=v)
    i = i + 1
  end
end
task_t4()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")
    plt.close(fig)

# Verify today's HOME matches the HOME used in NB4d A2 collection.
A2_TCP = [0.054, -0.274, 0.426]   # from NB4d A2 first sample
A2_Q   = [-1.377, -1.051, -1.391, -1.963, -4.352, -1.635]

drift_xyz_mm = np.sqrt(sum((tcp0[i]-A2_TCP[i])**2 for i in range(3))) * 1000
drift_q_max  = max(abs(q0[i]-A2_Q[i]) for i in range(6))
print(f"\nDrift vs NB4d A2 HOME:")
print(f"  TCP xyz drift: {drift_xyz_mm:.1f} mm")
print(f"  max joint drift: {drift_q_max:.4f} rad ({np.degrees(drift_q_max):.2f} deg)")
if drift_xyz_mm > 30 or drift_q_max > 0.05:
    print(f"  WARNING: HOME has drifted significantly from NB4d A2 session.")
    print(f"  Free-drive back to: TCP={A2_TCP}, q={A2_Q}")
    proceed = input("  Type 'yes' to continue anyway, anything else to abort: ").strip().lower()
    if proceed != "yes":
        raise RuntimeError("HOME drift too large. Aborting; restore HOME and rerun.")
else:
    print(f"  HOME matches NB4d A2 session (within tolerance). PROCEED.")

# Save the matched HOME as the T4 reference position.
home_ref = {
    "tcp": list(map(float, tcp0)),
    "q":   list(map(float, q0)),
    "saved_by": NOTEBOOK,
    "saved_at": datetime.now().isoformat(),
    "note":     "Canonical T4 HOME for all matched-session T4 collections. Verify drift vs this on every T4 run.",
}
with open(HOME_REF, "w") as f:
    json.dump(home_ref, f, indent=2)
print(f"  Saved canonical HOME reference: {HOME_REF}")

print(f"\nNotebook: {NOTEBOOK}")
print(f"Payload:  {PAYLOAD_PHYSICAL_KG}kg physical, {PAYLOAD_PENDANT_KG}kg pendant")
print(f"Cycles:   script={N_SCRIPT}, accept={N_ACCEPT}, hz={HZ}")

ts_start = datetime.now()
fname = f"UR5_{TASK}_healthy_{SESSION}_{N_SCRIPT}cyc_{HZ}hz_{ts_start.strftime('%Y%m%d_%H%M%S')}.h5"
fpath = LAB_DATA / fname
print(f"Output: {fpath}")

max_sec = N_SCRIPT * 30 + 60
nmax    = int(max_sec * HZ)

q     = np.zeros((nmax, 6), dtype=np.float32)
qd    = np.zeros((nmax, 6), dtype=np.float32)
cur   = np.zeros((nmax, 6), dtype=np.float32)
tgt_q = np.zeros((nmax, 6), dtype=np.float32)
tcp   = np.zeros((nmax, 6), dtype=np.float32)
ts    = np.zeros(nmax, dtype=np.float64)

print("Logging started; sending URScript in 1.5s...")
time.sleep(1.5)

t0 = time.perf_counter()
send_urscript(build_t4(N_SCRIPT))
print("URScript sent. Robot moving.\n")

n = 0
last_motion = t0
moving_seen = False
period = 1.0 / HZ
next_t = t0 + period

while n < nmax:
    now = time.perf_counter()
    if now < next_t:
        time.sleep(max(0.0, next_t - now))
    next_t += period

    try:
        q[n]     = rtde_r.getActualQ()
        qd[n]    = rtde_r.getActualQd()
        cur[n]   = rtde_r.getActualCurrent()
        tgt_q[n] = rtde_r.getTargetQ()
        tcp[n]   = rtde_r.getActualTCPPose()
        ts[n]    = time.perf_counter() - t0
    except Exception as e:
        print(f"RTDE read error at n={n}: {e}")
        break

    if np.any(np.abs(qd[n]) > 0.01):
        last_motion = ts[n]
        moving_seen = True

    if moving_seen and (ts[n] - last_motion) > 5.0:
        print(f"Motion stopped >5s at t={ts[n]:.1f}s. Ending capture.")
        n += 1
        break

    if ts[n] > max_sec:
        print(f"Max recording time {max_sec}s reached.")
        n += 1
        break

    n += 1

n_rec = n
duration = ts[n_rec - 1] if n_rec > 0 else 0.0
print(f"Captured {n_rec} samples over {duration:.1f}s")

q     = q[:n_rec]
qd    = qd[:n_rec]
cur   = cur[:n_rec]
tgt_q = tgt_q[:n_rec]
tcp   = tcp[:n_rec]
ts    = ts[:n_rec]

home_pos = tcp[0, :3]
dist_mm = np.sqrt(np.sum((tcp[:, :3] - home_pos)**2, axis=1)) * 1000.0

n_base = min(int(2.0 * HZ), len(dist_mm) // 10)
noise  = np.median(np.abs(np.diff(dist_mm[:n_base]))) if n_base > 10 else 0.5
home_r = max(10.0, 3.0 * noise)
far_r  = 30.0
min_cycle_sec = 18.0

cycle_num = np.zeros(len(tcp), dtype=np.int32)
c = 0
was_far = False
last_t = 0.0
for i in range(1, len(tcp)):
    if dist_mm[i] > far_r:
        was_far = True
    if was_far and dist_mm[i] < home_r and (ts[i] - last_t) > min_cycle_sec:
        c += 1
        last_t = ts[i]
        was_far = False
    cycle_num[i] = c

n_cycles_seen = int(cycle_num.max())
print(f"Cycles segmented: {n_cycles_seen} (home_r={home_r:.1f}mm)")

with h5py.File(str(fpath), "w") as f:
    f.create_dataset("actual_q",        data=q,     compression="gzip")
    f.create_dataset("actual_qd",       data=qd,    compression="gzip")
    f.create_dataset("actual_current",  data=cur,   compression="gzip")
    f.create_dataset("target_q",        data=tgt_q, compression="gzip")
    f.create_dataset("actual_TCP_pose", data=tcp,   compression="gzip")
    f.create_dataset("timestamp",       data=ts,    compression="gzip")
    f.create_dataset("cycle_number",    data=cycle_num, compression="gzip")
    f.attrs["task"]                = TASK
    f.attrs["condition"]           = "healthy"
    f.attrs["session"]             = SESSION
    f.attrs["notebook"]            = NOTEBOOK
    f.attrs["hz"]                  = HZ
    f.attrs["n_script"]            = N_SCRIPT
    f.attrs["n_accept"]            = N_ACCEPT
    f.attrs["duration_sec"]        = round(float(duration), 1)
    f.attrs["payload_pendant_kg"]  = PAYLOAD_PENDANT_KG
    f.attrs["payload_physical_kg"] = PAYLOAD_PHYSICAL_KG
    f.attrs["v"]                   = V
    f.attrs["a"]                   = A
    f.attrs["rot_rad"]             = ROT
    f.attrs["rot_deg"]             = float(np.degrees(ROT))
    f.attrs["home_r_mm"]           = float(home_r)
    f.attrs["far_r_mm"]            = float(far_r)
    f.attrs["n_cycles_seen"]       = n_cycles_seen
    f.attrs["timestamp_iso"]       = ts_start.isoformat()
    f.attrs["matched_to"]          = "NB4d_A2_session"

print(f"Saved: {fpath} ({fpath.stat().st_size / 1e6:.1f} MB)")

log_exists = DATASET_LOG.exists()
with open(DATASET_LOG, "a", newline="") as f:
    w = csv.writer(f)
    if not log_exists:
        w.writerow(["timestamp_iso", "notebook", "task", "condition", "severity",
                    "filename", "n_samples", "n_cycles_seen", "duration_sec", "hz",
                    "payload_pendant_kg", "payload_physical_kg", "v", "a", "rot_deg"])
    w.writerow([ts_start.isoformat(), NOTEBOOK, TASK, "healthy_session2", "",
                fname, n_rec, n_cycles_seen,
                round(duration,1), HZ, PAYLOAD_PENDANT_KG, PAYLOAD_PHYSICAL_KG,
                V, A, float(np.degrees(ROT))])
print(f"Appended to {DATASET_LOG}")

fig, axs = plt.subplots(3, 1, figsize=FIG_TALL, sharex=True)

for j in range(6):
    axs[0].plot(ts, cur[:, j], label=f"J{j+1}", linewidth=0.4)
axs[0].set_ylabel("Current (A)")
axs[0].set_title(f"{TASK} healthy session2 — joint currents")
axs[0].legend(ncol=6, loc="upper right", frameon=False)

axs[1].plot(ts, dist_mm, linewidth=0.5)
axs[1].axhline(home_r, linestyle="--", linewidth=0.4, label=f"home_r={home_r:.1f}mm")
axs[1].axhline(far_r,  linestyle="--", linewidth=0.4, label=f"far_r={far_r:.1f}mm")
axs[1].set_ylabel("|TCP - HOME| (mm)")
axs[1].legend(loc="upper right", frameon=False)

axs[2].plot(ts, cycle_num, linewidth=0.4)
axs[2].set_ylabel("Cycle #")
axs[2].set_xlabel("Time (s)")
axs[2].set_title(f"Cycles detected: {n_cycles_seen}")

plt.tight_layout()
save_fig(fig, f"FigS_{TASK}_healthy_session2_currents")

print(f"\n{NOTEBOOK} complete.")
print(f"Cycles seen: {n_cycles_seen}, target accept: {N_ACCEPT}")
print(f"File: {fpath}")
print(f"\nThis file is the matched-HOME healthy reference for the NB4d A2 anomaly comparison.")

In [ ]:
# Baseline: 2.0 kg physical, 2.0 kg pendant (same as NB1d healthy)
# Anomalies: A2 added mass (0.5, 1.0, 2.0 kg ADDED on top of baseline 2.0 kg)
# Pendant stays at 2.0 kg throughout. The mismatch is the anomaly.
# 3 conditions x 30 cycles ~80 min including swap pauses

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
print(f"TCP: x={tcp0[0]:.3f}  y={tcp0[1]:.3f}  z={tcp0[2]:.3f}")
print()
print("T4 A2 anomaly: pendant stays at 2.0 kg throughout.")
print("You will be prompted to ADD weight to the gripper between conditions.")
print("Total physical mass progression: 2.5 -> 3.0 -> 4.0 kg")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_SINGLE = (90/25.4, 70/25.4)
FIG_DOUBLE = (180/25.4, 70/25.4)

C_HEALTHY = "#0072B2"
C_ANOMALY = "#D55E00"

ROOT        = Path(r"D:\Research\RCM")
LAB_DATA    = ROOT / "Lab_Data" / "T4_BinReorient" / "anomaly"
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LAB_DATA, LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB4d_T4_A2_Anomaly"
TASK     = "T4"
HZ       = 125
N_SCRIPT = 35   # script asks for 35, accept ~30
N_ACCEPT = 30

# Pendant declaration is FIXED at 2.0 kg for the entire notebook.
# The physical mass changes per condition. The mismatch is the anomaly.
PAYLOAD_PENDANT_KG = 2.0

V     = 0.06
A     = 0.25
SLEEP = 0.3
ROT   = 1.0472   # +60 deg about TCP Z, same as NB1d

SCRIPT_PORT = 30002

CONDITIONS = [
    {
        "anomaly": "A2",
        "severity": "0.5kg",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "payload_physical_kg": 2.5,
        "instruction": (
            "A2 ADDED MASS (0.5 kg added):\n"
            "  Current total in gripper: 2.0 kg.\n"
            "  ADD a 0.5 kg weight to the gripper (total now 2.5 kg).\n"
            "  Pendant stays at 2.0 kg. DO NOT change pendant.\n"
            "  Verify weight is centrally gripped, no slip.\n"
            "  Press Enter when ready."
        ),
    },
    {
        "anomaly": "A2",
        "severity": "1kg",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "payload_physical_kg": 3.0,
        "instruction": (
            "A2 ADDED MASS (1.0 kg added):\n"
            "  Current total in gripper: 2.5 kg.\n"
            "  REMOVE the 0.5 kg, ADD a 1.0 kg weight (total now 3.0 kg).\n"
            "  Pendant stays at 2.0 kg. DO NOT change pendant.\n"
            "  Verify weight is centrally gripped, no slip.\n"
            "  Press Enter when ready."
        ),
    },
    {
        "anomaly": "A2",
        "severity": "2kg",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "payload_physical_kg": 4.0,
        "instruction": (
            "A2 ADDED MASS (2.0 kg added):\n"
            "  Current total in gripper: 3.0 kg.\n"
            "  REMOVE the 1.0 kg, ADD a 2.0 kg weight (total now 4.0 kg).\n"
            "  Pendant stays at 2.0 kg. DO NOT change pendant.\n"
            "  Verify weight is centrally gripped, no slip.\n"
            "  Press Enter when ready."
        ),
    },
]

def send_urscript(script: str):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect((ROBOT_IP, SCRIPT_PORT))
    s.send((script + "\n").encode())
    s.close()

def build_t4(N):
    return f"""
def task_t4():
  local v  = {V}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above  = pose_trans(home, p[ 0.10, 0.00,  0.05, 0,0, 0])
  local pick        = pose_trans(home, p[ 0.10, 0.00, -0.05, 0,0, 0])
  local place_above = pose_trans(home, p[-0.10, 0.10,  0.05, 0,0, {ROT}])
  local place       = pose_trans(home, p[-0.10, 0.10, -0.05, 0,0, {ROT}])

  while (i < N):
    movel(pick_above,  a=a, v=v)
    movel(pick,        a=a, v=v*0.7)
    sleep(dt)
    movel(pick_above,  a=a, v=v*0.7)

    movel(place_above, a=a, v=v)
    movel(place,       a=a, v=v*0.7)
    sleep(dt)
    movel(place_above, a=a, v=v*0.7)

    movel(home, a=a, v=v)
    i = i + 1
  end
end
task_t4()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")
    plt.close(fig)

def run_condition(cond):
    print("\n" + "=" * 70)
    print(cond["instruction"])
    print("=" * 70)
    input(">>> ")

    # Verify HOME drift
    tcp_now = rtde_r.getActualTCPPose()
    print(f"  HOME check: x={tcp_now[0]:.3f} y={tcp_now[1]:.3f} z={tcp_now[2]:.3f}")
    drift = np.sqrt((tcp_now[0]-tcp0[0])**2 + (tcp_now[1]-tcp0[1])**2 + (tcp_now[2]-tcp0[2])**2)*1000
    print(f"  HOME drift from initial: {drift:.1f} mm")
    if drift > 20:
        print("  WARNING: HOME has drifted >20mm. Free-drive back to original HOME before continuing.")
        input("  Press Enter once HOME is restored... ")

    ts_start = datetime.now()
    cond_str = f"{cond['anomaly']}_{cond['severity']}"
    fname = f"UR5_{TASK}_{cond_str}_{cond['n_script']}cyc_{HZ}hz_{ts_start.strftime('%Y%m%d_%H%M%S')}.h5"
    fpath = LAB_DATA / fname
    print(f"  Output: {fpath}")

    max_sec = cond["n_script"] * 30 + 60
    nmax    = int(max_sec * HZ)

    q     = np.zeros((nmax, 6), dtype=np.float32)
    qd    = np.zeros((nmax, 6), dtype=np.float32)
    cur   = np.zeros((nmax, 6), dtype=np.float32)
    tgt_q = np.zeros((nmax, 6), dtype=np.float32)
    tcp   = np.zeros((nmax, 6), dtype=np.float32)
    ts    = np.zeros(nmax, dtype=np.float64)

    print("  Logging started; sending URScript in 1.5s...")
    time.sleep(1.5)

    t0 = time.perf_counter()
    send_urscript(build_t4(cond["n_script"]))
    print("  URScript sent. Robot moving.")

    n = 0
    last_motion = t0
    moving_seen = False
    period = 1.0 / HZ
    next_t = t0 + period

    while n < nmax:
        now = time.perf_counter()
        if now < next_t:
            time.sleep(max(0.0, next_t - now))
        next_t += period

        try:
            q[n]     = rtde_r.getActualQ()
            qd[n]    = rtde_r.getActualQd()
            cur[n]   = rtde_r.getActualCurrent()
            tgt_q[n] = rtde_r.getTargetQ()
            tcp[n]   = rtde_r.getActualTCPPose()
            ts[n]    = time.perf_counter() - t0
        except Exception as e:
            print(f"  RTDE read error at n={n}: {e}")
            break

        if np.any(np.abs(qd[n]) > 0.01):
            last_motion = ts[n]
            moving_seen = True

        if moving_seen and (ts[n] - last_motion) > 5.0:
            print(f"  Motion stopped >5s at t={ts[n]:.1f}s. Ending capture.")
            n += 1
            break

        if ts[n] > max_sec:
            print(f"  Max recording time {max_sec}s reached.")
            n += 1
            break

        n += 1

    n_rec = n
    duration = ts[n_rec - 1] if n_rec > 0 else 0.0
    print(f"  Captured {n_rec} samples over {duration:.1f}s")

    q     = q[:n_rec]
    qd    = qd[:n_rec]
    cur   = cur[:n_rec]
    tgt_q = tgt_q[:n_rec]
    tcp   = tcp[:n_rec]
    ts    = ts[:n_rec]

    # Cycle segmentation
    home_pos = tcp[0, :3]
    dist_mm = np.sqrt(np.sum((tcp[:, :3] - home_pos)**2, axis=1)) * 1000.0

    n_base = min(int(2.0 * HZ), len(dist_mm) // 10)
    noise  = np.median(np.abs(np.diff(dist_mm[:n_base]))) if n_base > 10 else 0.5
    home_r = max(10.0, 3.0 * noise)
    far_r  = 30.0
    min_cycle_sec = 18.0

    cycle_num = np.zeros(len(tcp), dtype=np.int32)
    c = 0
    was_far = False
    last_t = 0.0
    for i in range(1, len(tcp)):
        if dist_mm[i] > far_r:
            was_far = True
        if was_far and dist_mm[i] < home_r and (ts[i] - last_t) > min_cycle_sec:
            c += 1
            last_t = ts[i]
            was_far = False
        cycle_num[i] = c

    n_cycles_seen = int(cycle_num.max())
    print(f"  Cycles segmented: {n_cycles_seen}")

    # Save HDF5
    with h5py.File(str(fpath), "w") as f:
        f.create_dataset("actual_q",        data=q,     compression="gzip")
        f.create_dataset("actual_qd",       data=qd,    compression="gzip")
        f.create_dataset("actual_current",  data=cur,   compression="gzip")
        f.create_dataset("target_q",        data=tgt_q, compression="gzip")
        f.create_dataset("actual_TCP_pose", data=tcp,   compression="gzip")
        f.create_dataset("timestamp",       data=ts,    compression="gzip")
        f.create_dataset("cycle_number",    data=cycle_num, compression="gzip")
        f.attrs["task"]                = TASK
        f.attrs["condition"]           = cond_str
        f.attrs["anomaly"]             = cond["anomaly"]
        f.attrs["severity"]            = cond["severity"]
        f.attrs["notebook"]            = NOTEBOOK
        f.attrs["hz"]                  = HZ
        f.attrs["n_script"]            = cond["n_script"]
        f.attrs["n_accept"]            = cond["n_accept"]
        f.attrs["duration_sec"]        = round(float(duration), 1)
        f.attrs["payload_pendant_kg"]  = PAYLOAD_PENDANT_KG
        f.attrs["payload_physical_kg"] = cond["payload_physical_kg"]
        f.attrs["v"]                   = V
        f.attrs["a"]                   = A
        f.attrs["rot_rad"]             = ROT
        f.attrs["rot_deg"]             = float(np.degrees(ROT))
        f.attrs["home_r_mm"]           = float(home_r)
        f.attrs["far_r_mm"]            = float(far_r)
        f.attrs["n_cycles_seen"]       = n_cycles_seen
        f.attrs["timestamp_iso"]       = ts_start.isoformat()

    print(f"  Saved: {fpath} ({fpath.stat().st_size / 1e6:.1f} MB)")

    # Append to log
    log_exists = DATASET_LOG.exists()
    with open(DATASET_LOG, "a", newline="") as f:
        w = csv.writer(f)
        if not log_exists:
            w.writerow(["timestamp_iso", "notebook", "task", "condition", "severity",
                        "filename", "n_samples", "n_cycles_seen", "duration_sec", "hz",
                        "payload_pendant_kg", "payload_physical_kg", "v", "a", "rot_deg"])
        w.writerow([ts_start.isoformat(), NOTEBOOK, TASK, cond_str, cond["severity"],
                    fname, n_rec, n_cycles_seen,
                    round(duration,1), HZ, PAYLOAD_PENDANT_KG, cond["payload_physical_kg"],
                    V, A, float(np.degrees(ROT))])

    return {
        "cond_str": cond_str,
        "fpath": fpath,
        "n_cycles_seen": n_cycles_seen,
        "duration": duration,
        "cur": cur,
        "ts": ts,
        "cycle_num": cycle_num,
    }

results = []
for cond in CONDITIONS:
    res = run_condition(cond)
    results.append(res)

print("\n" + "=" * 70)
print("All A2 conditions complete.")
print("=" * 70)
for r in results:
    print(f"  {r['cond_str']:20s} cycles={r['n_cycles_seen']:3d}  duration={r['duration']:.1f}s  file={r['fpath'].name}")

print("\nFINAL STEP: remove the 2.0 kg added weight from the gripper.")
print("Restore baseline: 2.0 kg gripped, 2.0 kg pendant.")

# Find latest healthy file
HEALTHY_DIR = ROOT / "Lab_Data" / "T4_BinReorient" / "healthy"
healthy_files = sorted(HEALTHY_DIR.glob("UR5_T4_healthy_*.h5"))
if len(healthy_files) == 0:
    print("WARNING: no T4 healthy file found, skipping QC overlay.")
else:
    h_path = healthy_files[-1]
    print(f"\nQC overlay vs healthy: {h_path.name}")
    with h5py.File(str(h_path), "r") as f:
        h_cur = f["actual_current"][:]
        h_ts  = f["timestamp"][:]
        h_cyc = f["cycle_number"][:]

    fig, axs = plt.subplots(3, 2, figsize=FIG_DOUBLE, sharex=False)
    axs = axs.flatten()
    h_idx = (h_cyc >= 20) & (h_cyc <= 22)
    h_t   = h_ts[h_idx] - h_ts[h_idx][0] if h_idx.sum() > 0 else None

    for j in range(6):
        ax = axs[j]
        if h_idx.sum() > 0:
            ax.plot(h_t, h_cur[h_idx, j], color=C_HEALTHY, linewidth=0.4,
                    alpha=0.8, label="healthy")
        for r, label in zip(results, ["A2 0.5kg", "A2 1.0kg", "A2 2.0kg"]):
            mask = (r["cycle_num"] >= 1) & (r["cycle_num"] <= 3)
            if mask.sum() > 0:
                t_rel = r["ts"][mask] - r["ts"][mask][0]
                ax.plot(t_rel, r["cur"][mask, j], linewidth=0.4,
                        alpha=0.7, label=label)
        ax.set_title(f"J{j+1}")
        ax.set_ylabel("A")
        ax.set_xlabel("t (s)")
        if j == 0:
            ax.legend(loc="upper right", frameon=False, fontsize=4)
    plt.tight_layout()
    save_fig(fig, "FigS_T4_A2_comparison")

print(f"\n{NOTEBOOK} complete.")

In [ ]:
# Anomaly: duct tape wrapped around J2 housing (additional Coulomb friction on J2)
# Severities: 7 wraps, 14 wraps
# Gripper: 2.0 kg (unchanged). Pendant: 2.0 kg (unchanged).
# 2 conditions x 35 cycles ~50 min including 1 wrap pause

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
q0   = rtde_r.getActualQ()
print(f"TCP[0]: x={tcp0[0]:+.3f} y={tcp0[1]:+.3f} z={tcp0[2]:+.3f}")
print(f"q[0]:   J1={q0[0]:+.3f} J2={q0[1]:+.3f} J3={q0[2]:+.3f} J4={q0[3]:+.3f} J5={q0[4]:+.3f} J6={q0[5]:+.3f}")
print()
print("T4 A3 anomaly: duct tape on J2 housing.")
print("Pendant 2.0 kg, gripper 2.0 kg baseline (no added mass).")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_DOUBLE = (180/25.4, 70/25.4)
C_HEALTHY = "#0072B2"
C_ANOMALY = "#D55E00"

ROOT        = Path(r"D:\Research\RCM")
LAB_DATA    = ROOT / "Lab_Data" / "T4_BinReorient" / "anomaly"
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
HOME_REF    = ROOT / "Lab_Data" / "T4_BinReorient" / "T4_HOME_reference.json"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LAB_DATA, LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB5d_T4_A3_Anomaly"
TASK     = "T4"
HZ       = 125
N_SCRIPT = 35
N_ACCEPT = 30

PAYLOAD_PENDANT_KG  = 2.0
PAYLOAD_PHYSICAL_KG = 2.0

V     = 0.06
A     = 0.25
SLEEP = 0.3
ROT   = 1.0472   # +60 deg about TCP Z

SCRIPT_PORT = 30002

CONDITIONS = [
    {
        "anomaly":  "A3",
        "severity": "7wraps",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "instruction": (
            "A3 J2 FRICTION (7 wraps duct tape):\n"
            "  Wrap duct tape 7 times around J2 housing where it meets the upper arm.\n"
            "  Use duct tape only (NOT electrical tape).\n"
            "  Wraps should be tight, fully contacting housing surface.\n"
            "  Gripper stays 2.0 kg baseline. Pendant stays 2.0 kg.\n"
            "  Press Enter when 7 wraps applied."
        ),
    },
    {
        "anomaly":  "A3",
        "severity": "14wraps",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "instruction": (
            "A3 J2 FRICTION (14 wraps duct tape):\n"
            "  ADD 7 more wraps on top of the existing 7 (total 14).\n"
            "  Same location on J2 housing.\n"
            "  Gripper stays 2.0 kg baseline. Pendant stays 2.0 kg.\n"
            "  Press Enter when 14 wraps applied."
        ),
    },
]

def send_urscript(script: str):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect((ROBOT_IP, SCRIPT_PORT))
    s.send((script + "\n").encode())
    s.close()

def build_t4(N):
    return f"""
def task_t4():
  local v  = {V}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above  = pose_trans(home, p[ 0.10, 0.00,  0.05, 0,0, 0])
  local pick        = pose_trans(home, p[ 0.10, 0.00, -0.05, 0,0, 0])
  local place_above = pose_trans(home, p[-0.10, 0.10,  0.05, 0,0, {ROT}])
  local place       = pose_trans(home, p[-0.10, 0.10, -0.05, 0,0, {ROT}])

  while (i < N):
    movel(pick_above,  a=a, v=v)
    movel(pick,        a=a, v=v*0.7)
    sleep(dt)
    movel(pick_above,  a=a, v=v*0.7)

    movel(place_above, a=a, v=v)
    movel(place,       a=a, v=v*0.7)
    sleep(dt)
    movel(place_above, a=a, v=v*0.7)

    movel(home, a=a, v=v)
    i = i + 1
  end
end
task_t4()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")
    plt.close(fig)

if not HOME_REF.exists():
    raise RuntimeError(f"Canonical HOME reference not found at {HOME_REF}. "
                       f"Run NB1d_T4_Healthy_session2.py first to establish HOME anchor.")

with open(HOME_REF, "r") as f:
    home_ref = json.load(f)

ref_tcp = np.array(home_ref["tcp"])
ref_q   = np.array(home_ref["q"])

drift_xyz_mm = np.sqrt(np.sum((np.array(tcp0[:3]) - ref_tcp[:3])**2)) * 1000
drift_q_max  = np.max(np.abs(np.array(q0) - ref_q))

print(f"\nHOME drift vs canonical T4 reference:")
print(f"  reference TCP: x={ref_tcp[0]:+.3f} y={ref_tcp[1]:+.3f} z={ref_tcp[2]:+.3f}")
print(f"  current TCP:   x={tcp0[0]:+.3f} y={tcp0[1]:+.3f} z={tcp0[2]:+.3f}")
print(f"  TCP xyz drift: {drift_xyz_mm:.2f} mm")
print(f"  max joint drift: {drift_q_max:.4f} rad ({np.degrees(drift_q_max):.3f} deg)")

if drift_xyz_mm > 30 or drift_q_max > 0.05:
    print(f"\n  WARNING: HOME has drifted significantly from canonical T4 reference.")
    print(f"  Free-drive back to: TCP={list(map(lambda x: round(x,3), ref_tcp))}")
    print(f"                       q={list(map(lambda x: round(x,3), ref_q))}")
    proceed = input("\n  Type 'yes' to continue anyway, anything else to abort: ").strip().lower()
    if proceed != "yes":
        raise RuntimeError("HOME drift too large. Restore HOME and rerun.")
else:
    print(f"\n  HOME matches canonical T4 reference (within tolerance). PROCEED.")

print(f"\nNotebook: {NOTEBOOK}")
print(f"Conditions: {len(CONDITIONS)}  ({[c['severity'] for c in CONDITIONS]})")

def run_condition(cond):
    print("\n" + "=" * 70)
    print(cond["instruction"])
    print("=" * 70)
    input(">>> ")

    # Recheck HOME after physical perturbation (tape may have nudged the arm)
    tcp_now = rtde_r.getActualTCPPose()
    q_now   = rtde_r.getActualQ()
    drift_xyz = np.sqrt(np.sum((np.array(tcp_now[:3]) - ref_tcp[:3])**2)) * 1000
    drift_q   = np.max(np.abs(np.array(q_now) - ref_q))
    print(f"  HOME post-tape check: TCP drift={drift_xyz:.2f} mm, max joint={np.degrees(drift_q):.3f} deg")
    if drift_xyz > 30 or drift_q > 0.05:
        print(f"  WARNING: HOME drifted during tape application. Free-drive back.")
        input(f"  Press Enter once HOME restored... ")

    ts_start = datetime.now()
    cond_str = f"{cond['anomaly']}_{cond['severity']}"
    fname = f"UR5_{TASK}_{cond_str}_{cond['n_script']}cyc_{HZ}hz_{ts_start.strftime('%Y%m%d_%H%M%S')}.h5"
    fpath = LAB_DATA / fname
    print(f"  Output: {fpath}")

    max_sec = cond["n_script"] * 30 + 60
    nmax    = int(max_sec * HZ)

    q     = np.zeros((nmax, 6), dtype=np.float32)
    qd    = np.zeros((nmax, 6), dtype=np.float32)
    cur   = np.zeros((nmax, 6), dtype=np.float32)
    tgt_q = np.zeros((nmax, 6), dtype=np.float32)
    tcp   = np.zeros((nmax, 6), dtype=np.float32)
    ts    = np.zeros(nmax, dtype=np.float64)

    print("  Logging started; sending URScript in 1.5s...")
    time.sleep(1.5)

    t0 = time.perf_counter()
    send_urscript(build_t4(cond["n_script"]))
    print("  URScript sent. Robot moving.")

    n = 0
    last_motion = t0
    moving_seen = False
    period = 1.0 / HZ
    next_t = t0 + period

    while n < nmax:
        now = time.perf_counter()
        if now < next_t:
            time.sleep(max(0.0, next_t - now))
        next_t += period

        try:
            q[n]     = rtde_r.getActualQ()
            qd[n]    = rtde_r.getActualQd()
            cur[n]   = rtde_r.getActualCurrent()
            tgt_q[n] = rtde_r.getTargetQ()
            tcp[n]   = rtde_r.getActualTCPPose()
            ts[n]    = time.perf_counter() - t0
        except Exception as e:
            print(f"  RTDE read error at n={n}: {e}")
            break

        if np.any(np.abs(qd[n]) > 0.01):
            last_motion = ts[n]
            moving_seen = True

        if moving_seen and (ts[n] - last_motion) > 5.0:
            print(f"  Motion stopped >5s at t={ts[n]:.1f}s. Ending capture.")
            n += 1
            break

        if ts[n] > max_sec:
            print(f"  Max recording time {max_sec}s reached.")
            n += 1
            break

        n += 1

    n_rec = n
    duration = ts[n_rec - 1] if n_rec > 0 else 0.0
    print(f"  Captured {n_rec} samples over {duration:.1f}s")

    q     = q[:n_rec]
    qd    = qd[:n_rec]
    cur   = cur[:n_rec]
    tgt_q = tgt_q[:n_rec]
    tcp   = tcp[:n_rec]
    ts    = ts[:n_rec]

    # Cycle segmentation
    home_pos = tcp[0, :3]
    dist_mm = np.sqrt(np.sum((tcp[:, :3] - home_pos)**2, axis=1)) * 1000.0

    n_base = min(int(2.0 * HZ), len(dist_mm) // 10)
    noise  = np.median(np.abs(np.diff(dist_mm[:n_base]))) if n_base > 10 else 0.5
    home_r = max(10.0, 3.0 * noise)
    far_r  = 30.0
    min_cycle_sec = 18.0

    cycle_num = np.zeros(len(tcp), dtype=np.int32)
    c = 0
    was_far = False
    last_t = 0.0
    for i in range(1, len(tcp)):
        if dist_mm[i] > far_r:
            was_far = True
        if was_far and dist_mm[i] < home_r and (ts[i] - last_t) > min_cycle_sec:
            c += 1
            last_t = ts[i]
            was_far = False
        cycle_num[i] = c

    n_cycles_seen = int(cycle_num.max())
    print(f"  Cycles segmented: {n_cycles_seen}")

    # Save HDF5
    with h5py.File(str(fpath), "w") as f:
        f.create_dataset("actual_q",        data=q,     compression="gzip")
        f.create_dataset("actual_qd",       data=qd,    compression="gzip")
        f.create_dataset("actual_current",  data=cur,   compression="gzip")
        f.create_dataset("target_q",        data=tgt_q, compression="gzip")
        f.create_dataset("actual_TCP_pose", data=tcp,   compression="gzip")
        f.create_dataset("timestamp",       data=ts,    compression="gzip")
        f.create_dataset("cycle_number",    data=cycle_num, compression="gzip")
        f.attrs["task"]                = TASK
        f.attrs["condition"]           = cond_str
        f.attrs["anomaly"]             = cond["anomaly"]
        f.attrs["severity"]            = cond["severity"]
        f.attrs["notebook"]            = NOTEBOOK
        f.attrs["hz"]                  = HZ
        f.attrs["n_script"]            = cond["n_script"]
        f.attrs["n_accept"]            = cond["n_accept"]
        f.attrs["duration_sec"]        = round(float(duration), 1)
        f.attrs["payload_pendant_kg"]  = PAYLOAD_PENDANT_KG
        f.attrs["payload_physical_kg"] = PAYLOAD_PHYSICAL_KG
        f.attrs["v"]                   = V
        f.attrs["a"]                   = A
        f.attrs["rot_rad"]             = ROT
        f.attrs["rot_deg"]             = float(np.degrees(ROT))
        f.attrs["home_r_mm"]           = float(home_r)
        f.attrs["far_r_mm"]            = float(far_r)
        f.attrs["n_cycles_seen"]       = n_cycles_seen
        f.attrs["timestamp_iso"]       = ts_start.isoformat()
        f.attrs["matched_to"]          = "T4_HOME_reference"

    print(f"  Saved: {fpath} ({fpath.stat().st_size / 1e6:.1f} MB)")

    log_exists = DATASET_LOG.exists()
    with open(DATASET_LOG, "a", newline="") as f:
        w = csv.writer(f)
        if not log_exists:
            w.writerow(["timestamp_iso", "notebook", "task", "condition", "severity",
                        "filename", "n_samples", "n_cycles_seen", "duration_sec", "hz",
                        "payload_pendant_kg", "payload_physical_kg", "v", "a", "rot_deg"])
        w.writerow([ts_start.isoformat(), NOTEBOOK, TASK, cond_str, cond["severity"],
                    fname, n_rec, n_cycles_seen,
                    round(duration,1), HZ, PAYLOAD_PENDANT_KG, PAYLOAD_PHYSICAL_KG,
                    V, A, float(np.degrees(ROT))])

    return {
        "cond_str": cond_str,
        "fpath": fpath,
        "n_cycles_seen": n_cycles_seen,
        "duration": duration,
        "cur": cur,
        "ts": ts,
        "cycle_num": cycle_num,
    }

results = []
for cond in CONDITIONS:
    res = run_condition(cond)
    results.append(res)

print("\n" + "=" * 70)
print("All A3 conditions complete.")
print("=" * 70)
for r in results:
    print(f"  {r['cond_str']:20s} cycles={r['n_cycles_seen']:3d}  duration={r['duration']:.1f}s  file={r['fpath'].name}")

print("\nFINAL STEP: REMOVE all duct tape from J2 before next experiment.")
print("Restore baseline: 2.0 kg gripped, 2.0 kg pendant, no tape.")

HEALTHY_DIR = ROOT / "Lab_Data" / "T4_BinReorient" / "healthy"
healthy_files = sorted(HEALTHY_DIR.glob("UR5_T4_healthy_session2_*.h5"))
if len(healthy_files) == 0:
    print("WARNING: no T4 session2 healthy file found, skipping QC overlay.")
else:
    h_path = healthy_files[-1]
    print(f"\nQC overlay vs healthy: {h_path.name}")
    with h5py.File(str(h_path), "r") as f:
        h_cur = f["actual_current"][:]
        h_ts  = f["timestamp"][:]
        h_cyc = f["cycle_number"][:]

    fig, axs = plt.subplots(3, 2, figsize=FIG_DOUBLE, sharex=False)
    axs = axs.flatten()
    h_idx = (h_cyc >= 5) & (h_cyc <= 7)
    h_t   = h_ts[h_idx] - h_ts[h_idx][0] if h_idx.sum() > 0 else None

    for j in range(6):
        ax = axs[j]
        if h_idx.sum() > 0:
            ax.plot(h_t, h_cur[h_idx, j], color=C_HEALTHY, linewidth=0.4,
                    alpha=0.8, label="healthy")
        for r, label in zip(results, ["A3 7wraps", "A3 14wraps"]):
            mask = (r["cycle_num"] >= 5) & (r["cycle_num"] <= 7)
            if mask.sum() > 0:
                t_rel = r["ts"][mask] - r["ts"][mask][0]
                ax.plot(t_rel, r["cur"][mask, j], linewidth=0.4,
                        alpha=0.7, label=label)
        ax.set_title(f"J{j+1}")
        ax.set_ylabel("A")
        ax.set_xlabel("t (s)")
        if j == 1:
            ax.legend(loc="upper right", frameon=False, fontsize=4)
    plt.tight_layout()
    save_fig(fig, "FigS_T4_A3_comparison")

print(f"\n{NOTEBOOK} complete.")

In [ ]:
# Anomaly: rigid extension attached to gripper, displacing effective TCP
# Severities: 20 mm, 50 mm, 100 mm extension
# Gripper: 2.0 kg baseline (unchanged). Pendant: 2.0 kg (unchanged).
# 3 conditions x 35 cycles ~50 min including 2 swap pauses

from rtde_receive import RTDEReceiveInterface

ROBOT_IP = "192.168.35.82"
rtde_r = RTDEReceiveInterface(ROBOT_IP)
print("Connected:", rtde_r.isConnected())

tcp0 = rtde_r.getActualTCPPose()
q0   = rtde_r.getActualQ()
print(f"TCP[0]: x={tcp0[0]:+.3f} y={tcp0[1]:+.3f} z={tcp0[2]:+.3f}")
print(f"q[0]:   J1={q0[0]:+.3f} J2={q0[1]:+.3f} J3={q0[2]:+.3f} J4={q0[3]:+.3f} J5={q0[4]:+.3f} J6={q0[5]:+.3f}")
print()
print("T4 A5 anomaly: rigid extension on gripper.")
print("Confirm: all duct tape removed from J2.")
print("Pendant 2.0 kg, gripper 2.0 kg baseline.")

import socket, time, json, csv
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime

matplotlib.rcParams.update({
    "font.family":       "Arial",
    "font.size":         6,
    "axes.titlesize":    7,
    "axes.labelsize":    6,
    "xtick.labelsize":   5,
    "ytick.labelsize":   5,
    "legend.fontsize":   5,
    "figure.titlesize":  7,
    "axes.linewidth":    0.5,
    "xtick.major.width": 0.4,
    "ytick.major.width": 0.4,
    "xtick.major.size":  2,
    "ytick.major.size":  2,
    "lines.linewidth":   0.4,
    "savefig.dpi":       1200,
    "savefig.bbox":      "tight",
    "savefig.pad_inches": 0.02,
    "pdf.fonttype":      42,
    "ps.fonttype":       42,
})

FIG_DOUBLE = (180/25.4, 70/25.4)
C_HEALTHY = "#0072B2"

ROOT        = Path(r"D:\Research\RCM")
LAB_DATA    = ROOT / "Lab_Data" / "T4_BinReorient" / "anomaly"
LOGS_DIR    = ROOT / "Logs"
DATASET_LOG = ROOT / "Lab_Data" / "dataset_log.csv"
HOME_REF    = ROOT / "Lab_Data" / "T4_BinReorient" / "T4_HOME_reference.json"
FIG_SUP     = ROOT / "Manuscript_Data" / "Figures" / "Supplementary"
for d in [LAB_DATA, LOGS_DIR, FIG_SUP]:
    d.mkdir(parents=True, exist_ok=True)

NOTEBOOK = "NB6d_T4_A5_Anomaly"
TASK     = "T4"
HZ       = 125
N_SCRIPT = 35
N_ACCEPT = 30

PAYLOAD_PENDANT_KG  = 2.0
PAYLOAD_PHYSICAL_KG = 2.0

V     = 0.06
A     = 0.25
SLEEP = 0.3
ROT   = 1.0472   # +60 deg about TCP Z

SCRIPT_PORT = 30002

CONDITIONS = [
    {
        "anomaly":  "A5",
        "severity": "20mm",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "instruction": (
            "A5 TCP OFFSET (20 mm extension):\n"
            "  Attach the 20 mm rigid extension to the gripper.\n"
            "  Same orientation as used in T3 A5 collection (NB6).\n"
            "  Gripper still holds 2.0 kg baseline.\n"
            "  Pendant stays 2.0 kg. DO NOT change pendant.\n"
            "  Press Enter when 20 mm extension attached."
        ),
    },
    {
        "anomaly":  "A5",
        "severity": "50mm",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "instruction": (
            "A5 TCP OFFSET (50 mm extension):\n"
            "  REMOVE the 20 mm extension, attach the 50 mm extension.\n"
            "  Gripper still holds 2.0 kg baseline.\n"
            "  Pendant stays 2.0 kg.\n"
            "  Press Enter when 50 mm extension attached."
        ),
    },
    {
        "anomaly":  "A5",
        "severity": "100mm",
        "n_script": N_SCRIPT,
        "n_accept": N_ACCEPT,
        "instruction": (
            "A5 TCP OFFSET (100 mm extension):\n"
            "  REMOVE the 50 mm extension, attach the 100 mm extension.\n"
            "  Gripper still holds 2.0 kg baseline.\n"
            "  Pendant stays 2.0 kg.\n"
            "  Press Enter when 100 mm extension attached."
        ),
    },
]

def send_urscript(script: str):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.connect((ROBOT_IP, SCRIPT_PORT))
    s.send((script + "\n").encode())
    s.close()

def build_t4(N):
    return f"""
def task_t4():
  local v  = {V}
  local a  = {A}
  local dt = {SLEEP}
  local N  = {N}
  local i  = 0

  local home = get_actual_tcp_pose()

  local pick_above  = pose_trans(home, p[ 0.10, 0.00,  0.05, 0,0, 0])
  local pick        = pose_trans(home, p[ 0.10, 0.00, -0.05, 0,0, 0])
  local place_above = pose_trans(home, p[-0.10, 0.10,  0.05, 0,0, {ROT}])
  local place       = pose_trans(home, p[-0.10, 0.10, -0.05, 0,0, {ROT}])

  while (i < N):
    movel(pick_above,  a=a, v=v)
    movel(pick,        a=a, v=v*0.7)
    sleep(dt)
    movel(pick_above,  a=a, v=v*0.7)

    movel(place_above, a=a, v=v)
    movel(place,       a=a, v=v*0.7)
    sleep(dt)
    movel(place_above, a=a, v=v*0.7)

    movel(home, a=a, v=v)
    i = i + 1
  end
end
task_t4()
"""

def save_fig(fig, name):
    fig.savefig(FIG_SUP / f"{name}.png")
    fig.savefig(FIG_SUP / f"{name}.pdf")
    plt.close(fig)

if not HOME_REF.exists():
    raise RuntimeError(f"Canonical HOME reference not found at {HOME_REF}.")

with open(HOME_REF, "r") as f:
    home_ref = json.load(f)

ref_tcp = np.array(home_ref["tcp"])
ref_q   = np.array(home_ref["q"])

drift_xyz_mm = np.sqrt(np.sum((np.array(tcp0[:3]) - ref_tcp[:3])**2)) * 1000
drift_q_max  = np.max(np.abs(np.array(q0) - ref_q))

print(f"\nHOME drift vs canonical T4 reference:")
print(f"  TCP xyz drift: {drift_xyz_mm:.2f} mm")
print(f"  max joint drift: {np.degrees(drift_q_max):.3f} deg")
if drift_xyz_mm > 30 or drift_q_max > 0.05:
    print(f"\n  WARNING: HOME has drifted. Free-drive back.")
    proceed = input("  Type 'yes' to continue anyway, anything else to abort: ").strip().lower()
    if proceed != "yes":
        raise RuntimeError("HOME drift too large.")
else:
    print(f"  HOME matches canonical T4 reference. PROCEED.")

print(f"\nNotebook: {NOTEBOOK}")
print(f"Conditions: {[c['severity'] for c in CONDITIONS]}")

def run_condition(cond):
    print("\n" + "=" * 70)
    print(cond["instruction"])
    print("=" * 70)
    input(">>> ")

    tcp_now = rtde_r.getActualTCPPose()
    q_now   = rtde_r.getActualQ()
    drift_xyz = np.sqrt(np.sum((np.array(tcp_now[:3]) - ref_tcp[:3])**2)) * 1000
    drift_q   = np.max(np.abs(np.array(q_now) - ref_q))
    print(f"  HOME post-attach check: TCP drift={drift_xyz:.2f} mm, max joint={np.degrees(drift_q):.3f} deg")
    if drift_xyz > 30 or drift_q > 0.05:
        print(f"  WARNING: HOME drifted during attachment. Free-drive back.")
        input(f"  Press Enter once HOME restored... ")

    ts_start = datetime.now()
    cond_str = f"{cond['anomaly']}_{cond['severity']}"
    fname = f"UR5_{TASK}_{cond_str}_{cond['n_script']}cyc_{HZ}hz_{ts_start.strftime('%Y%m%d_%H%M%S')}.h5"
    fpath = LAB_DATA / fname
    print(f"  Output: {fpath}")

    max_sec = cond["n_script"] * 30 + 60
    nmax    = int(max_sec * HZ)

    q     = np.zeros((nmax, 6), dtype=np.float32)
    qd    = np.zeros((nmax, 6), dtype=np.float32)
    cur   = np.zeros((nmax, 6), dtype=np.float32)
    tgt_q = np.zeros((nmax, 6), dtype=np.float32)
    tcp   = np.zeros((nmax, 6), dtype=np.float32)
    ts    = np.zeros(nmax, dtype=np.float64)

    print("  Logging started; sending URScript in 1.5s...")
    time.sleep(1.5)

    t0 = time.perf_counter()
    send_urscript(build_t4(cond["n_script"]))
    print("  URScript sent. Robot moving.")

    n = 0
    last_motion = t0
    moving_seen = False
    period = 1.0 / HZ
    next_t = t0 + period

    while n < nmax:
        now = time.perf_counter()
        if now < next_t:
            time.sleep(max(0.0, next_t - now))
        next_t += period

        try:
            q[n]     = rtde_r.getActualQ()
            qd[n]    = rtde_r.getActualQd()
            cur[n]   = rtde_r.getActualCurrent()
            tgt_q[n] = rtde_r.getTargetQ()
            tcp[n]   = rtde_r.getActualTCPPose()
            ts[n]    = time.perf_counter() - t0
        except Exception as e:
            print(f"  RTDE read error at n={n}: {e}")
            break

        if np.any(np.abs(qd[n]) > 0.01):
            last_motion = ts[n]
            moving_seen = True

        if moving_seen and (ts[n] - last_motion) > 5.0:
            print(f"  Motion stopped >5s at t={ts[n]:.1f}s. Ending capture.")
            n += 1
            break

        if ts[n] > max_sec:
            print(f"  Max recording time {max_sec}s reached.")
            n += 1
            break

        n += 1

    n_rec = n
    duration = ts[n_rec - 1] if n_rec > 0 else 0.0
    print(f"  Captured {n_rec} samples over {duration:.1f}s")

    q     = q[:n_rec]
    qd    = qd[:n_rec]
    cur   = cur[:n_rec]
    tgt_q = tgt_q[:n_rec]
    tcp   = tcp[:n_rec]
    ts    = ts[:n_rec]

    home_pos = tcp[0, :3]
    dist_mm = np.sqrt(np.sum((tcp[:, :3] - home_pos)**2, axis=1)) * 1000.0

    n_base = min(int(2.0 * HZ), len(dist_mm) // 10)
    noise  = np.median(np.abs(np.diff(dist_mm[:n_base]))) if n_base > 10 else 0.5
    home_r = max(10.0, 3.0 * noise)
    far_r  = 30.0
    min_cycle_sec = 18.0

    cycle_num = np.zeros(len(tcp), dtype=np.int32)
    c = 0
    was_far = False
    last_t = 0.0
    for i in range(1, len(tcp)):
        if dist_mm[i] > far_r:
            was_far = True
        if was_far and dist_mm[i] < home_r and (ts[i] - last_t) > min_cycle_sec:
            c += 1
            last_t = ts[i]
            was_far = False
        cycle_num[i] = c

    n_cycles_seen = int(cycle_num.max())
    print(f"  Cycles segmented: {n_cycles_seen}")

    with h5py.File(str(fpath), "w") as f:
        f.create_dataset("actual_q",        data=q,     compression="gzip")
        f.create_dataset("actual_qd",       data=qd,    compression="gzip")
        f.create_dataset("actual_current",  data=cur,   compression="gzip")
        f.create_dataset("target_q",        data=tgt_q, compression="gzip")
        f.create_dataset("actual_TCP_pose", data=tcp,   compression="gzip")
        f.create_dataset("timestamp",       data=ts,    compression="gzip")
        f.create_dataset("cycle_number",    data=cycle_num, compression="gzip")
        f.attrs["task"]                = TASK
        f.attrs["condition"]           = cond_str
        f.attrs["anomaly"]             = cond["anomaly"]
        f.attrs["severity"]            = cond["severity"]
        f.attrs["notebook"]            = NOTEBOOK
        f.attrs["hz"]                  = HZ
        f.attrs["n_script"]            = cond["n_script"]
        f.attrs["n_accept"]            = cond["n_accept"]
        f.attrs["duration_sec"]        = round(float(duration), 1)
        f.attrs["payload_pendant_kg"]  = PAYLOAD_PENDANT_KG
        f.attrs["payload_physical_kg"] = PAYLOAD_PHYSICAL_KG
        f.attrs["v"]                   = V
        f.attrs["a"]                   = A
        f.attrs["rot_rad"]             = ROT
        f.attrs["rot_deg"]             = float(np.degrees(ROT))
        f.attrs["home_r_mm"]           = float(home_r)
        f.attrs["far_r_mm"]            = float(far_r)
        f.attrs["n_cycles_seen"]       = n_cycles_seen
        f.attrs["timestamp_iso"]       = ts_start.isoformat()
        f.attrs["matched_to"]          = "T4_HOME_reference"

    print(f"  Saved: {fpath} ({fpath.stat().st_size / 1e6:.1f} MB)")

    log_exists = DATASET_LOG.exists()
    with open(DATASET_LOG, "a", newline="") as f:
        w = csv.writer(f)
        if not log_exists:
            w.writerow(["timestamp_iso", "notebook", "task", "condition", "severity",
                        "filename", "n_samples", "n_cycles_seen", "duration_sec", "hz",
                        "payload_pendant_kg", "payload_physical_kg", "v", "a", "rot_deg"])
        w.writerow([ts_start.isoformat(), NOTEBOOK, TASK, cond_str, cond["severity"],
                    fname, n_rec, n_cycles_seen,
                    round(duration,1), HZ, PAYLOAD_PENDANT_KG, PAYLOAD_PHYSICAL_KG,
                    V, A, float(np.degrees(ROT))])

    return {
        "cond_str": cond_str,
        "fpath": fpath,
        "n_cycles_seen": n_cycles_seen,
        "duration": duration,
        "cur": cur,
        "ts": ts,
        "cycle_num": cycle_num,
    }

results = []
for cond in CONDITIONS:
    res = run_condition(cond)
    results.append(res)

print("\n" + "=" * 70)
print("All A5 conditions complete.")
print("=" * 70)
for r in results:
    print(f"  {r['cond_str']:20s} cycles={r['n_cycles_seen']:3d}  duration={r['duration']:.1f}s  file={r['fpath'].name}")

print("\nFINAL STEP: REMOVE the 100 mm extension from the gripper.")
print("Restore baseline: 2.0 kg gripped, 2.0 kg pendant, no extension.")

HEALTHY_DIR = ROOT / "Lab_Data" / "T4_BinReorient" / "healthy"
healthy_files = sorted(HEALTHY_DIR.glob("UR5_T4_healthy_session2_*.h5"))
if len(healthy_files) > 0:
    h_path = healthy_files[-1]
    print(f"\nQC overlay vs healthy: {h_path.name}")
    with h5py.File(str(h_path), "r") as f:
        h_cur = f["actual_current"][:]
        h_ts  = f["timestamp"][:]
        h_cyc = f["cycle_number"][:]

    fig, axs = plt.subplots(3, 2, figsize=FIG_DOUBLE, sharex=False)
    axs = axs.flatten()
    h_idx = (h_cyc >= 5) & (h_cyc <= 7)
    h_t   = h_ts[h_idx] - h_ts[h_idx][0] if h_idx.sum() > 0 else None

    for j in range(6):
        ax = axs[j]
        if h_idx.sum() > 0:
            ax.plot(h_t, h_cur[h_idx, j], color=C_HEALTHY, linewidth=0.4, alpha=0.8, label="healthy")
        for r, label in zip(results, ["A5 20mm", "A5 50mm", "A5 100mm"]):
            mask = (r["cycle_num"] >= 5) & (r["cycle_num"] <= 7)
            if mask.sum() > 0:
                t_rel = r["ts"][mask] - r["ts"][mask][0]
                ax.plot(t_rel, r["cur"][mask, j], linewidth=0.4, alpha=0.7, label=label)
        ax.set_title(f"J{j+1}")
        ax.set_ylabel("A")
        ax.set_xlabel("t (s)")
        if j == 0:
            ax.legend(loc="upper right", frameon=False, fontsize=4)
    plt.tight_layout()
    save_fig(fig, "FigS_T4_A5_comparison")

print(f"\n{NOTEBOOK} complete.")